In [78]:
import numpy as np
import pandas as pd
import glob 
import math
from pathlib import Path

TAB_LINKER = " & "
TAB_END = "\\\\"
CHECK_SYMBOL = "\\checkmark"
CROSS_SYMBOL = "\\texttimes"

In [ ]:
def get_ListOfDF(directory:str, verbose:int = 0):
    # Read all the results tables in the directory
    csv_files = glob.glob(directory)

    # Sort the tables by name
    csv_files = sorted(csv_files)

    # Read all the csv files from the directory, and save we save them in a list
    dataframes = [pd.read_csv(f) for f in csv_files]
    names_frames_complete = [Path(csv).stem.split('.csv')[0] for csv in csv_files] 

    #We print the name of the readed tables if it's requiered
    if verbose in [-1, 0]: 
        print(names_frames_complete)
        print(names_frames_complete)
    if verbose in [0,1]: print("Readed path: ", directory)

    # We print the tables readed
    if verbose == 0:
        for i in range(0, len(csv_files)):
            print(f"|{i}| The table has been read it from the directory: {csv_files[i]}")
        print("-"*80)
    if verbose in [-1,0,1]:
        print(f"{len(dataframes)} tables has been readed.")

    return dataframes, names_frames_complete 


#Directory of the results original
directory_original = "csvs/results/robust/original/*.csv"
directory_simplest = "csvs/results/robust/simplest/*.csv"
directory_addedLS = "csvs/results/robust/addedLS/*.csv"


#Read all the results of the dataframes
dataframes_orig, names_frames = get_ListOfDF(directory_original, 0)
dataframes_LS, names_frames = get_ListOfDF(directory_addedLS, 0)
dataframes_SP, names_frames = get_ListOfDF(directory_simplest, 0)

['WATSON_FACTOR_1', 'WATSON_FACTOR_2', 'WATSON_FACTOR_3', 'WATSON_FACTOR_4', 'WATSON_FACTOR_5', 'WATSON_FACTOR_6']
['WATSON_FACTOR_1', 'WATSON_FACTOR_2', 'WATSON_FACTOR_3', 'WATSON_FACTOR_4', 'WATSON_FACTOR_5', 'WATSON_FACTOR_6']
Readed path:  csvs/results/robust/original/*.csv
|0| The table has been read it from the directory: csvs/results/robust/original/WATSON_FACTOR_1.csv
|1| The table has been read it from the directory: csvs/results/robust/original/WATSON_FACTOR_2.csv
|2| The table has been read it from the directory: csvs/results/robust/original/WATSON_FACTOR_3.csv
|3| The table has been read it from the directory: csvs/results/robust/original/WATSON_FACTOR_4.csv
|4| The table has been read it from the directory: csvs/results/robust/original/WATSON_FACTOR_5.csv
|5| The table has been read it from the directory: csvs/results/robust/original/WATSON_FACTOR_6.csv
--------------------------------------------------------------------------------
6 tables has been readed.
['WATSON_FACTO

In [ ]:
julia robust_main.jl --problem 1-cutest-sif/WATSON.SIF --seed 42 --NTRIES 6 --nIters 10000 --show_info --subdirectory simplest --varP 31
julia robust_main.jl --problem 1-cutest-sif/WATSON.SIF --seed 42 --NTRIES 6 --nIters 10000 --show_info --modifierH eigen --subdirectory original --varP 31
julia robust_main.jl --problem 1-cutest-sif/WATSON.SIF --seed 42 --NTRIES 6 --nIters 10000 --show_info --useLS --modifierH eigen --subdirectory addedLS --varP 31


In [80]:
def create_matrix_results(dataframes_list: list[pd.DataFrame])->np.ndarray:
    """
    # Definition
    This function create an np.ndarray element class. Each element in the dataframe
    has the results from the the three methods, whose have the following order:

    1. NAMGM/AMG
    2. NAMGM/Queue
    3. NAMGM/Random

    And the order of the columns is 

    1. Convergence 
    2. Number of iterations taken
    3. Execution time
    4. Last gradient norm of the sequence

    Observe that we are not longer measuring the 'iterations per second'. 
    
    # Inputs
    The big difference is that the creation of each matrix is based in one problem and we not have
    to modify the order of the variables (this is the big difference btw `robust_tables` and `creation_tables`).
    
    - dataframes_list: list[pd.dataframes] - The list of dataframes of the different factors for the problem


    # Output

    - matrix_results: np.ndarray - Numpy array with the results for the Configuration/Problem

    """
    # We have 5 methods and 5 variables, then in total we have N \times 5 elements in the dataframe
    result_matrix = np.zeros((len(dataframes_list), 12)) #(Number of factors, Variables (4 variables, but 3 methods))

    #Picking of the information of each method (picking in rows)
    for i in range(0, len(dataframes_list)):
        results_variables_per_factor: np.ndarray = dataframes_list[i].to_numpy().flatten()
        result_matrix[i,:] = results_variables_per_factor
    return result_matrix

#Matrices with the results
matrix_simplest = create_matrix_results(dataframes_SP)  
matrix_original = create_matrix_results(dataframes_orig) 
matrix_addedLS = create_matrix_results(dataframes_LS)  

In [97]:
def construct_cell_color(color1:str, alpha1:int, color2:str =None, alpha2:int=None)->str:
    """# Usage
    
    Function to construct the color cell on a LaTeX table. If color2 and alpha 2 are
    not provided, it still works.
    
    ## Input:
        - ``color1``: string - First (or main) color of the cell
        - ``alpha1``:   int  - Transparency (or percentage of the first color) 
        - ``color1``(optional): string - Second color of the cell
        - ``alpha2``(optional):   int  - Transparency of the combined color 

    ## Output:
        - ``s``: string - LaTeX command to color the cell
    """
    final_color:str = f"{color1}!{alpha1}"
    if color2:
        final_color += f"!{color2}"
        if alpha2:
            final_color += f"!{alpha2}"
    return f"\\cellcolor{{{final_color}}}"

def __elementTable__(information:np.ndarray, times_symbol:str = "\\times", 
                     add_cell_color:bool = True, convergence_color:str = "ForestGreen", not_Convergence_color:str = "BrickRed", cell_alpha:int=10):
    """Suppose that some one give us the results for a factor, and all the methods are flatted in order."""
    s = ""
    #Each variable has its own format
    convergence_flags = [bool(information[i]) for i in range(0, information.shape[0], 4)]
    iterations = [int(information[i]) for i in range(1, information.shape[0], 4)]
    exe_time = [f"{information[i]: .4f}" for i in range(2, information.shape[0], 4)]

    gradient_norms = [information[i].item() for i in range(3, information.shape[0], 4)]
    g_formatted = []
    for g in gradient_norms:
        exponent = int(math.floor(math.log10(abs(g))))
        decimal = g / (10 ** exponent)
        g_formatted.append(f"${decimal: .3f}{times_symbol} 10^{{{int(exponent)}}} $")
    
    for i in range(0, 3):
        c = CHECK_SYMBOL if convergence_flags[i] else CROSS_SYMBOL            
        if add_cell_color:
            c+= construct_cell_color(convergence_color, cell_alpha) if convergence_flags[i] else construct_cell_color(not_Convergence_color, cell_alpha)
        
        s += c + TAB_LINKER + str(iterations[i]) + TAB_LINKER + exe_time[i] + TAB_LINKER + g_formatted[i]
        s += TAB_LINKER if i!=2 else TAB_END
    return s

def __tableBody__(factors_matrices: list[np.ndarray], times_symbol="\\times", 
                  add_cell_color:bool = True, convergence_color:str = "ForestGreen", not_Convergence_color:str = "BrickRed", cell_alpha:int=10)->None:
    
    #Assertions to garantie the correct working of this function
    assert 0<=cell_alpha<=100 and type(cell_alpha)==int, "the alpha of the cell must be an integer and in the interval [0, 100]"
    assert len(factors_matrices)==3, "The list of matrices must have the tree configurations [SIMPLEST, ORIGINAL, WithLS]"
    
    #Row creation (Move first over the f)
    for f in range(factors_matrices[0].shape[0]):
        print(f"\\multirow{{3}}{{*}}{{$10^{{{f}}}$}}" + TAB_LINKER + __elementTable__(factors_matrices[0][f], times_symbol=times_symbol))
        print("" + TAB_LINKER + __elementTable__(factors_matrices[1][f], times_symbol=times_symbol))
        print("" + TAB_LINKER + __elementTable__(factors_matrices[2][f], times_symbol=times_symbol))
        print("\\hline") if f!=factors_matrices[0].shape[0]-1 else print("")
    return None 


In [98]:
#Print the resulting table
list_matrices = [matrix_simplest, matrix_original, matrix_simplest]

__tableBody__(list_matrices, "\\cdot")

\multirow{3}{*}{$10^{0}$} & \texttimes\cellcolor{BrickRed!10} & 1000 &  0.3444 & $ 8.686\cdot 10^{-5} $ & \texttimes\cellcolor{BrickRed!10} & 1000 &  0.1339 & $ 9.787\cdot 10^{-4} $ & \texttimes\cellcolor{BrickRed!10} & 1000 &  0.0499 & $ 1.279\cdot 10^{-2} $\\
 & \texttimes\cellcolor{BrickRed!10} & 1000 &  0.3579 & $ 9.822\cdot 10^{-5} $ & \texttimes\cellcolor{BrickRed!10} & 1000 &  0.1487 & $ 1.550\cdot 10^{-3} $ & \texttimes\cellcolor{BrickRed!10} & 1000 &  0.0668 & $ 1.172\cdot 10^{-2} $\\
 & \texttimes\cellcolor{BrickRed!10} & 1000 &  0.3444 & $ 8.686\cdot 10^{-5} $ & \texttimes\cellcolor{BrickRed!10} & 1000 &  0.1339 & $ 9.787\cdot 10^{-4} $ & \texttimes\cellcolor{BrickRed!10} & 1000 &  0.0499 & $ 1.279\cdot 10^{-2} $\\
\hline
\multirow{3}{*}{$10^{1}$} & \texttimes\cellcolor{BrickRed!10} & 1000 &  0.0555 & $ 8.686\cdot 10^{-5} $ & \texttimes\cellcolor{BrickRed!10} & 1000 &  0.0507 & $ 9.787\cdot 10^{-4} $ & \texttimes\cellcolor{BrickRed!10} & 1000 &  0.0500 & $ 9.925\cdot 10^{-3}